In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder # <------------------ Esto es nuevo :)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.compose import ColumnTransformer # <------------------ Esto es nuevo :)
from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline 
from sklearn.pipeline import Pipeline # <------------------ Esto es nuevo :)

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score


# Para guardar el modelo
import pickle

In [16]:
df = pd.read_csv('./data/titanic_clean.csv')

In [17]:
df['Age'] = df['Age'].fillna(df['Age'].median())

Preprocesamiento

A continuación crearemos un objeto especial que usaremos en nuestro pipeline. Este objeto definirá los pasos de preprocesamiento que se ejecutarán tanto en la fase de entrenamiento como en la fase de inferencia.

In [18]:
# Definir preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), ['Sex', 'Embarked']),  # OneHotEncoder para columnas categóricas
        ('age', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Age']),
        ('fare', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Fare'])
    ],
    remainder='passthrough'  # Mantener otras columnas sin cambios
)

¿Qué es OneHotEncoder?

En notebooks anteriores usamos LabelEncoder. No profundicemos en las diferencias técnicas. Para propósitos de nuestro proyecto OneHotEncoder y LabelEncoder hacen lo mismo. Tenemos que usar OneHotEncoder porque LabelEncoder no funciona con Pipelines. 

Definición de modelos

Nuestro diccionario de modelos tiene un cambio pequeño:

In [19]:
# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(n_neighbors=3),
        'parametros': {
            'model__n_neighbors': [3, 5, 7]
        }
    }
}

Los hiperparámetros llevan ahora un prefijo 'model__' .
División de datos

In [20]:
# División de datos
X = df.drop(['Survived'], axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

Ningún cambio significativo en esta etapa.

Variables auxiliares

In [21]:
# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

Ciclo for de GridSearch

El contenido de nuestro ciclo for sí cambiará un poco:

In [22]:
# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    # Crear pipeline para el modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', MinMaxScaler()),  # MinMaxScaler se aplica a todas las columnas después del preprocesamiento
        ('model', info_modelo['modelo'])  # Modelo placeholder
    ])
    
    # Inicializar GridSearchCV con el pipeline y los hiperparámetros
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)
    
    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)
    
    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)
    
    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_
    
    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

Mostrar resultados y guardar modelo

Lo último no sufre ningún cambio:

In [23]:
# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

# Guardar el mejor modelo con pickle
with open('pipeline.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

Rendimiento de los modelos de clasificación
                             Modelo  Precisión
0  Clasificador K-Nearest Neighbors       0.83
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.83
